# 11 · Wind check: pilots vs the weather model

The wind pilots measured in their thermals (the drift of their circles, notebook 08) against ERA5's wind **at the same place, hour and height** (10 m, 100 m and the pressure levels of notebook 07c, interpolated to the height of each measurement).

Measurements are pooled by site, chip, hour and height band, and scored separately in each band:

| Band | Height above the ground | What it checks |
|---|---|---|
| near the ground | below 400 m | the wind already shaped by the terrain: what WindNinja or the CFD surrogate has to get right (ERA5 does not resolve it) |
| middle | 400–1000 m | |
| free air | above 1000 m, or above the chip's highest ground | the prevailing wind: the input a terrain-flow model takes from the weather model |

If the free air agrees and the near-the-ground band does not, the error is the terrain's; if the free air already disagrees, a terrain model fed with that wind cannot be right either. Run it after notebook 07c (pressure levels); without them only the lowest heights can be compared. A few minutes.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "thermals.parquet")), "Run notebook 08 first: database/thermals.parquet is missing"
subprocess.run(["pip", "install", "-q", "lightgbm"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Compare

Vector error and direction error are medians; speed bias > 0 means pilots measured more wind than the model.

In [ ]:
run(f"{REPO}/scripts/wind_check.py", DB)

## 4. Charts

In [ ]:
import matplotlib.pyplot as plt, numpy as np
g = pd.read_parquet(os.path.join(DB, "models", "wind_check", "groups.parquet"))
bands = ["near the ground", "middle", "free air"]
fig, ax = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)
for a, b in zip(ax, bands):
    d = g[g["band"] == b]
    a.scatter(d["model_speed"], d["pilot_speed"], s=6, alpha=0.3, color="#2a78d6")
    m = float(np.nanpercentile(g[["model_speed", "pilot_speed"]], 99)) if len(g) else 1
    a.plot([0, m], [0, m], color="#c3c2b7", lw=1)
    a.set(title=f"{b} ({len(d):,} groups)", xlabel="ERA5 wind (m/s)")
    a.spines[["top", "right"]].set_visible(False)
ax[0].set_ylabel("pilots' wind (m/s)")
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(6, 3.6))
for b, c in zip(bands, ["#c3c2b7", "#86b6ef", "#2a78d6"]):
    ax.hist(g.loc[g["band"] == b, "dir_error"].dropna(), bins=np.arange(0, 181, 10), histtype="step", lw=2, color=c, label=b)
ax.set(xlabel="direction error (degrees, model wind 3 m/s or more)", ylabel="groups")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()